# Korean MCQ Benchmark

> **Original source:** [hyogrin/rhoai-lmeval-builder-lab](https://github.com/hyogrin/rhoai-lmeval-builder-lab)  
> **Author:** hyogrin  
> **Adapted by:** [gymnatics/RHOAI-Toolkit](https://github.com/gymnatics/RHOAI-Toolkit) for dynamic cluster configuration

In [ ]:
# Auto-configure environment for this cluster
import os
try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".env"), override=True)
except ImportError:
    pass

NAMESPACE = os.environ.get("NAMESPACE", "lmeval-demo")
MODEL_NAME = os.environ.get("MODEL_NAME", "")
BASE_URL = os.environ.get("BASE_URL", "")
EVALHUB_URL = os.environ.get("EVALHUB_URL", "http://evalhub.redhat-ods-applications.svc:8080")

# Auto-detect model if not set
if not MODEL_NAME or not BASE_URL:
    import subprocess
    try:
        result = subprocess.run(
            ["oc", "get", "inferenceservice", "-n", NAMESPACE, "--no-headers", "-o", "custom-columns=NAME:.metadata.name"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0 and result.stdout.strip():
            detected = result.stdout.strip().split("\n")[0]
            if not MODEL_NAME:
                MODEL_NAME = detected
            if not BASE_URL:
                BASE_URL = f"https://{detected}-predictor.{NAMESPACE}.svc.cluster.local:8443/v1"
            print(f"Auto-detected model: {MODEL_NAME}")
    except Exception:
        pass

print(f"Namespace:  {NAMESPACE}")
print(f"Model:      {MODEL_NAME}")
print(f"Base URL:   {BASE_URL}")
print(f"EvalHub:    {EVALHUB_URL}")

## Step 1: Configuration

In [ ]:
import subprocess

# Auth token: prefer injected env var (evalhub-service SA token), fall back to oc whoami -t
EVALHUB_AUTH_TOKEN = os.environ.get("EVALHUB_AUTH_TOKEN", "")
if not EVALHUB_AUTH_TOKEN:
    _r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    EVALHUB_AUTH_TOKEN = _r.stdout.strip() if _r.returncode == 0 else None
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "https://mlflow.redhat-ods-applications.svc:8443")
LIMIT = int(os.environ.get("LIMIT", "5"))

print(f"Namespace:       {NAMESPACE}")
print(f"Model Name:      {MODEL_NAME}")
print(f"Model Endpoint:  {BASE_URL}")
print(f"EvalHub URL:     {EVALHUB_URL}")
print(f"MLflow URI:      {MLFLOW_TRACKING_URI}")
print(f"Sample Limit:    {LIMIT}")

## Step 2: Initialize the EvalHub Client

In [ ]:
from evalhub import (
    SyncEvalHubClient,
    ModelConfig,
    BenchmarkConfig,
    JobSubmissionRequest,
    ExperimentConfig,
    ExperimentTag,
    JobStatus,
)

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

model = ModelConfig(
    url=BASE_URL,
    name=MODEL_NAME,
)

print(f"EvalHub client connected: {EVALHUB_URL}")
print(f"Model target:             {model.name} @ {model.url}")

## Step 3: Discover Available Korean Benchmarks

Query EvalHub for benchmarks available through the `lm_evaluation_harness` provider and filter for Korean tasks.

In [ ]:
# NOTE: The Korean MCQ provider YAML and ../adapters/ directory are from
# the upstream repo: https://github.com/hyogrin/rhoai-lmeval-builder-lab
# Clone that repo if you need the provider definition files.
import yaml, pathlib, httpx

KOREAN_PROVIDER_YAML = pathlib.Path("../adapters/korean-mcq/provider.yaml")
KOREAN_PROVIDER_NAME = "Korean MCQ Evaluation"
KOREAN_PROVIDER_ID = None

for p in client.providers.list():
    if p.name == KOREAN_PROVIDER_NAME:
        KOREAN_PROVIDER_ID = p.resource.id
        print(f"Korean MCQ provider already registered (id={KOREAN_PROVIDER_ID})")
        break

if not KOREAN_PROVIDER_ID and KOREAN_PROVIDER_YAML.exists():
    raw = KOREAN_PROVIDER_YAML.read_text().replace("${NAMESPACE}", NAMESPACE)
    provider_def = yaml.safe_load(raw)
    resp = httpx.post(
        f"{EVALHUB_URL}/api/v1/evaluations/providers",
        headers={
            "Authorization": f"Bearer {EVALHUB_AUTH_TOKEN}",
            "Content-Type": "application/json",
            "X-Tenant": NAMESPACE,
        },
        json=provider_def, verify=False, timeout=10,
    )
    if resp.status_code in (200, 201):
        data = resp.json()
        KOREAN_PROVIDER_ID = data["resource"]["id"]
        print(f"Korean MCQ provider registered (id={KOREAN_PROVIDER_ID}, benchmarks={len(data.get('benchmarks', []))})")
    else:
        print(f"WARNING: Korean MCQ registration failed ({resp.status_code}): {resp.text[:200]}")

all_benchmarks = client.benchmarks.list()

korean_keywords = ["kmmlu", "kobest", "haerae", "klue", "korean", "ko_", "click", "hrm8k"]
korean_benchmarks = [
    bm for bm in all_benchmarks
    if any(kw in bm.id.lower() for kw in korean_keywords)
]

print(f"Total benchmarks available: {len(all_benchmarks)}")
print(f"Korean benchmarks found:    {len(korean_benchmarks)}")
print("=" * 70)
for bm in korean_benchmarks:
    metrics_str = ", ".join(bm.metrics[:3]) if bm.metrics else "N/A"
    print(f"  {bm.id:40s}  metrics=[{metrics_str}]")

---

## Evaluation 1: Single Korean Benchmark

Start with a single benchmark to verify the pipeline works end-to-end.

### 1-A: Submit the Job

In [ ]:
single_request = JobSubmissionRequest(
    name="kmmlu-eval",
    description="Korean MMLU benchmark - single task smoke test",
    tags=["korean", "kmmlu", "smoke-test"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="kmmlu",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"temperature": 0.0, "max_tokens": 16, "limit": LIMIT},
        ),
    ],
    experiment=ExperimentConfig(
        name="korean-kmmlu-eval",
        tags=[
            ExperimentTag(key="language", value="korean"),
            ExperimentTag(key="benchmark_suite", value="kmmlu"),
            ExperimentTag(key="evaluation_type", value="smoke-test"),
        ],
    ),
)

job = client.jobs.submit(single_request)

print(f"Job submitted successfully!")
print(f"  Job ID:        {job.id}")
print(f"  Name:          {job.name}")
print(f"  State:         {job.status.value}")
print(f"  MLflow Exp ID: {job.resource.mlflow_experiment_id or 'pending'}")

### 1-B: Monitor Progress

In [ ]:
import time
from datetime import datetime

TERMINAL_STATES = {
    JobStatus.COMPLETED,
    JobStatus.FAILED,
    JobStatus.CANCELLED,
}


def wait_for_job(client, job_id, poll_interval=10, max_wait=600):
    """Poll job status until terminal state or timeout."""
    start = time.time()
    print(f"Monitoring job {job_id}...")
    print("-" * 70)

    while time.time() - start < max_wait:
        status = client.jobs.get(job_id)
        state = status.state
        elapsed = int(time.time() - start)

        msg = ""
        if status.status and status.status.message:
            msg = f" | {status.status.message.message}"

        bm_info = ""
        if status.status and status.status.benchmarks:
            bm_states = [f"{b.id}={b.status.value}" for b in status.status.benchmarks]
            bm_info = f" | benchmarks: {', '.join(bm_states)}"

        print(f"  [{elapsed:>4d}s] {state.value:>16s}{msg}{bm_info}")

        if state in TERMINAL_STATES:
            break

        time.sleep(poll_interval)

    print("-" * 70)
    print(f"Final state: {state.value} (elapsed: {elapsed}s)")
    return status


completed_job = wait_for_job(client, job.id)

### 1-C: View Results

In [ ]:
def display_job_results(job):
    """Display evaluation results with MLflow links."""
    if not job.results:
        print("No results available.")
        return

    print("Evaluation Results")
    print("=" * 70)

    if job.results.mlflow_experiment_url:
        print(f"\n  MLflow Experiment: {job.results.mlflow_experiment_url}")

    for bm in job.results.benchmarks:
        print(f"\n  Benchmark: {bm.id}")
        print(f"  Provider:  {bm.provider_id}")
        if bm.mlflow_run_id:
            print(f"  MLflow Run: {bm.mlflow_run_id}")

        if bm.metrics:
            print(f"  Metrics:")
            for name, value in bm.metrics.items():
                if isinstance(value, float):
                    print(f"    {name:30s} = {value:.4f}")
                else:
                    print(f"    {name:30s} = {value}")
        else:
            print("  Metrics: (none)")


display_job_results(completed_job)

---

## Step 4: Job Management

List, inspect, and manage all evaluation jobs.

### List All Jobs

In [ ]:
jobs_list = client.jobs.list()

print(f"Total Jobs: {len(jobs_list)}")
print("=" * 90)
print(f"{'State':>16s}  {'Job ID':12s}  {'Name':30s}  {'Experiment':25s}  Benchmarks")
print("-" * 90)
for j in jobs_list:
    state = j.state.value
    exp = j.experiment.name if j.experiment else "N/A"
    bms = [b.id for b in j.benchmarks] if j.benchmarks else []
    print(f"{state:>16s}  {j.id[:12]:12s}  {j.name[:30]:30s}  {exp[:25]:25s}  {bms}")

### Inspect a Specific Job

Replace `JOB_ID` with the job you want to inspect.

In [ ]:
# Replace with actual job ID:
# JOB_ID = "your-job-id-here"
# inspected = client.jobs.get(JOB_ID)
# display_job_results(inspected)

---

## Step 5: Export Results

### Export to Markdown

In [ ]:
def collect_results_table(client):
    """Collect benchmark metrics from completed jobs into a comparison dict."""
    jobs_list = client.jobs.list()
    table = {}
    for j in jobs_list:
        if j.state != JobStatus.COMPLETED or not j.results:
            continue
        for bm in j.results.benchmarks:
            if bm.id not in table:
                table[bm.id] = {}
            table[bm.id][j.name] = bm.metrics
    return table


comparison = collect_results_table(client)


def results_to_markdown(comparison, title="EvalHub Benchmark Results"):
    """Convert comparison results to markdown table."""
    if not comparison:
        return "No results available."

    lines = [f"## {title}", ""]

    for benchmark_id, job_results in comparison.items():
        lines.append(f"### {benchmark_id}")
        lines.append("")

        all_metrics = set()
        for metrics in job_results.values():
            all_metrics.update(metrics.keys())
        all_metrics = sorted(all_metrics)

        job_names = sorted(job_results.keys())
        header = "| Metric | " + " | ".join(job_names) + " |"
        sep = "|---" + "|---" * len(job_names) + "|"
        lines.extend([header, sep])

        for metric in all_metrics:
            row = f"| {metric} |"
            for job_name in job_names:
                val = job_results.get(job_name, {}).get(metric, "-")
                if isinstance(val, float):
                    row += f" {val:.4f} |"
                else:
                    row += f" {val} |"
            lines.append(row)

        lines.append("")

    return "\n".join(lines)


md = results_to_markdown(comparison)
print(md)

### Save Results to JSON

In [ ]:
import json
from pathlib import Path

results_root = Path("../results")

jobs_list = client.jobs.list()
saved = 0
for j in jobs_list:
    if j.state != JobStatus.COMPLETED or not j.results:
        continue

    model_name = j.model.name if j.model else "unknown"
    model_dir = results_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    job_data = {
        "job_id": j.id,
        "name": j.name,
        "model": {"url": j.model.url, "name": j.model.name},
        "experiment": j.experiment.name if j.experiment else None,
        "benchmarks": [
            {
                "id": bm.id,
                "provider_id": bm.provider_id,
                "metrics": bm.metrics,
                "mlflow_run_id": bm.mlflow_run_id,
            }
            for bm in j.results.benchmarks
        ],
    }

    filename = f"{j.name}_{j.id[:8]}.json"
    output_path = model_dir / filename
    with open(output_path, "w") as f:
        json.dump(job_data, f, indent=2, default=str)
    print(f"Saved: {output_path}")
    saved += 1

print(f"\n{saved} result(s) saved to {results_root.resolve()}/<model-name>/")

---

## Summary

This notebook demonstrated a single Korean MCQ benchmark evaluation:

1. **Configuration** and EvalHub client setup
2. **Single benchmark** evaluation (KMMLU) with MLflow tracking
3. **Job management** -- list and inspect jobs
4. **Export** -- Markdown and JSON output

### Next Steps

- **2_summarize_results.ipynb** -- Aggregate results and generate HTML/Markdown reports
- **3_eval_hub_unified_benchmark/1_unified_benchmark.ipynb** -- Run multi-benchmark + GuideLLM unified evaluation
- **1_eval_hub_guidellm_benchmark/1_guidellm_benchmark.ipynb** -- Standalone GuideLLM performance profiling
